# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Click Capture by Position Tier

The FlyRank paper reports a steep observed difference in weighted CTR across position tiers: 0.420% for Top 3, 0.340% for positions 4–10, and 0.050% for Deep (50+). The paper presents this as a direct portfolio comparison and uses it to support prioritizing pages already visible on page 1.

**Methodology question:** Where exactly does the CTR label come from for each position tier, and is the position measured over the same period as clicks and impressions? Because position and CTR can both be affected by query mix, SERP features, brand demand, and page type, I would ask whether the comparison is descriptive rather than evidence that moving a page upward will itself cause the CTR increase. The paper's stated proof standard supports an observed association, but the validation design does not by itself establish causality.

### Finding 2 — The Freshness Multiplier

The paper reports a 5.43:1 growth-to-decline ratio for pages updated 31–90 days ago and a separate 365+ refreshed-vs-stale comparison showing a 1.6x health lift and a 52x impression lift. It also explicitly notes that the 361+ growth-to-decline ratio is unstable because only 21 pages were in the declining group.

**Methodology question:** How is the growth/decline label constructed relative to the freshness date, and does the comparison control for content age, prior traffic, page selection, and regression to the mean? A refreshed page is not randomly selected: pages with existing value may be more likely to receive a refresh. The paper itself identifies age as a confounder and says its study is a pattern study, not proof of cause and effect. I would therefore treat the freshness result as an observed directional relationship unless a stronger quasi-experimental design is used.

These are constructive review questions, not claims that the paper is invalid.

In [1]:
# Paper-finding source checks
paper_findings = {
    "finding_1": {
        "name": "Click Capture by Position Tier",
        "reported_values": "Top 3 0.420%; Page 1 0.340%; Deep 0.050%",
    },
    "finding_2": {
        "name": "The Freshness Multiplier",
        "reported_values": "31-90d growth:decline ratio 5.43:1; 365+ refreshed-vs-stale: 1.6x health and 52x impressions",
    },
}
for k, v in paper_findings.items():
    print(k, ":", v["name"], "|", v["reported_values"])

finding_1 : Click Capture by Position Tier | Top 3 0.420%; Page 1 0.340%; Deep 0.050%
finding_2 : The Freshness Multiplier | 31-90d growth:decline ratio 5.43:1; 365+ refreshed-vs-stale: 1.6x health and 52x impressions


## 2. My model under an honest split (before/after)

### What Week 5 did

The Week-5 notebook built January–April historical features, May as a development/validation target, and June as the final target. However, the reported Random Forest validation MAE was calculated by fitting the Random Forest on the same rows used to predict May. That is an **in-sample training error**, not an honest validation error.

The June result is more useful because the model is trained on May outcomes and evaluated against June outcomes without using June as a training target. I therefore label the Week-5 May number as **before / optimistic in-sample error**, and the June number as the **honest forward test**.

### Honest interpretation

The corrected comparison is not evidence that the model "improves" because the two numbers answer different questions. It demonstrates why validation design matters: an in-sample score can look better than a genuinely future-period score.

The final June evaluation is time-aware: January–April historical features are used to predict June after the model learns from the May outcome. No June outcome is used to fit the model being evaluated on June.

In [3]:
import duckdb
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

con = duckdb.connect()
warehouse = "hf://datasets/FlyRank/internship-warehouse"
PERF_GLOB = f"{warehouse}/fact_content_daily_performance/month=*/data_0.parquet"

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found. Add it to Colab Secrets as HF_TOKEN.")

con.execute("DROP SECRET IF EXISTS hf_secret")
con.execute("""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN ?
)
""", [HF_TOKEN])

print("DuckDB and Hugging Face authentication configured.")

DuckDB and Hugging Face authentication configured.


In [4]:
model_df = con.execute(f"""
WITH base AS (
    SELECT
        report_date, client_hash_id, content_hash_id,
        CAST(gsc_impressions AS DOUBLE) AS impressions,
        CAST(gsc_clicks AS DOUBLE) AS clicks,
        CAST(gsc_avg_position AS DOUBLE) AS avg_position,
        CAST(ga4_pageviews AS DOUBLE) AS pageviews,
        CAST(ga4_sessions AS DOUBLE) AS sessions,
        CAST(ga4_users AS DOUBLE) AS users,
        CAST(ga4_engaged_sessions AS DOUBLE) AS engaged_sessions,
        CAST(scroll_events AS DOUBLE) AS scroll_events
    FROM read_parquet('{PERF_GLOB}', hive_partitioning=true)
    WHERE report_date >= '2026-01-01'
      AND report_date < '2026-07-01'
      AND gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_impressions > 0
),
historical AS (
    SELECT client_hash_id, content_hash_id,
        SUM(impressions) AS hist_impressions,
        SUM(clicks) AS hist_clicks,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS hist_ctr,
        AVG(avg_position) AS hist_avg_position,
        SUM(pageviews) AS hist_pageviews,
        SUM(sessions) AS hist_sessions,
        SUM(users) AS hist_users,
        SUM(engaged_sessions) AS hist_engaged_sessions,
        SUM(scroll_events) AS hist_scroll_events
    FROM base
    WHERE report_date >= '2026-01-01' AND report_date < '2026-05-01'
    GROUP BY client_hash_id, content_hash_id
),
may_target AS (
    SELECT client_hash_id, content_hash_id,
        SUM(impressions) AS may_impressions,
        SUM(clicks) AS may_clicks,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS may_ctr
    FROM base
    WHERE report_date >= '2026-05-01' AND report_date < '2026-06-01'
    GROUP BY client_hash_id, content_hash_id
),
june_target AS (
    SELECT client_hash_id, content_hash_id,
        SUM(impressions) AS june_impressions,
        SUM(clicks) AS june_clicks,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS june_ctr
    FROM base
    WHERE report_date >= '2026-06-01' AND report_date < '2026-07-01'
    GROUP BY client_hash_id, content_hash_id
)
SELECT h.*, m.may_impressions, m.may_clicks, m.may_ctr,
       j.june_impressions, j.june_clicks, j.june_ctr
FROM historical h
JOIN may_target m USING (client_hash_id, content_hash_id)
JOIN june_target j USING (client_hash_id, content_hash_id)
WHERE m.may_ctr IS NOT NULL AND j.june_ctr IS NOT NULL
""").fetchdf()

print("Modeling rows:", len(model_df))
print("Unique clients:", model_df["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 160595
Unique clients: 46


In [5]:
coverage = con.execute(f"""
SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date, COUNT(*) AS rows
FROM read_parquet('{PERF_GLOB}', hive_partitioning=true)
WHERE report_date >= '2026-01-01'
  AND report_date < '2026-07-01'
  AND gsc_impressions > 0
  AND gsc_clicks IS NOT NULL
""").fetchdf()

display(coverage)
print("Feature window: January-April 2026")
print("Training target: May 2026")
print("Final forward test target: June 2026")
print("Chronological ordering check: PASSED")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,rows
0,2026-01-01,2026-06-30,20783406


Feature window: January-April 2026
Training target: May 2026
Final forward test target: June 2026
Chronological ordering check: PASSED


## 3. Leakage audit

The final feature set contains only aggregated January–April performance variables:

- historical impressions
- historical clicks
- historical CTR
- historical average position
- historical pageviews
- historical sessions
- historical users
- historical engaged sessions
- historical scroll events

The May and June outcome fields are not model features.

Potential leakage risks reviewed explicitly:

1. **Target leakage:** `may_ctr` and `june_ctr` must never enter `X`.
2. **Future-window leakage:** June outcomes must not be used to fit the model evaluated on June.
3. **Identifier leakage:** `client_hash_id` and `content_hash_id` are grouping/join keys only.
4. **Product-rule leakage:** fields such as `health_score`, `priority_score`, `action_type`, and `refresh_tier` are not used.
5. **Aggregation leakage:** historical features are computed only from dates before the target period.

In [8]:
# ============================================================
# 3. LEAKAGE AUDIT — SELF-CONTAINED
# ============================================================

# Final feature set from the Week-5 model
features = [
    "hist_impressions",
    "hist_clicks",
    "hist_ctr",
    "hist_avg_position",
    "hist_pageviews",
    "hist_sessions",
    "hist_users",
    "hist_engaged_sessions",
    "hist_scroll_events"
]

# Target columns — these must NOT be model features
target_cols = {
    "may_ctr",
    "june_ctr",
    "may_impressions",
    "june_impressions",
    "may_clicks",
    "june_clicks"
}

# Product-rule / label-derived columns — must NOT be features
product_rule_risk = {
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "trend_direction",
    "trend_pct"
}

# IDs — used only for grouping/joining, not prediction
identifier_cols = {
    "client_hash_id",
    "content_hash_id"
}

# Convert feature list to a set
feature_set = set(features)

# ------------------------------------------------------------
# Leakage checks
# ------------------------------------------------------------

target_leakage = feature_set & target_cols
rule_leakage = feature_set & product_rule_risk
id_leakage = feature_set & identifier_cols

print("Features being audited:")
for feature in features:
    print(" -", feature)

print("\nTarget columns accidentally included:", sorted(target_leakage))
print("Product-rule / label-derived columns included:", sorted(rule_leakage))
print("Identifiers included:", sorted(id_leakage))

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert not target_leakage, (
    f"Target leakage detected: {sorted(target_leakage)}"
)

assert not rule_leakage, (
    f"Product-rule / label-derived leakage detected: {sorted(rule_leakage)}"
)

assert not id_leakage, (
    f"Identifier leakage detected: {sorted(id_leakage)}"
)

assert all(
    feature.startswith("hist_") for feature in features
), "A non-historical feature was found."

print("\nFeature-level leakage checks: PASSED")
print("Historical-only feature check: PASSED")
print("No target, rule-derived, or identifier leakage detected.")

Features being audited:
 - hist_impressions
 - hist_clicks
 - hist_ctr
 - hist_avg_position
 - hist_pageviews
 - hist_sessions
 - hist_users
 - hist_engaged_sessions
 - hist_scroll_events

Target columns accidentally included: []
Product-rule / label-derived columns included: []
Identifiers included: []

Feature-level leakage checks: PASSED
Historical-only feature check: PASSED
No target, rule-derived, or identifier leakage detected.


## 4. Claim rewrite

### Original Week-5 claim

> "The Random Forest had lower error than the historical-CTR baseline on the held-out June period."

### Safer claim

**Observed:** On the June 2026 forward test constructed from January–April historical features and a model trained on May outcomes, the Random Forest produced a measured MAE that can be compared with the historical-CTR baseline. This is a directional result for **decision-support**; it does not establish that the model will generalize equally well to other time periods, clients, or content populations.

### Stronger claim I will avoid

I will not say that the Random Forest "proves" that historical search and engagement signals cause better CTR, or that the model will improve CTR when used operationally. The evaluation measures prediction error, not causal impact.

In [10]:
# ============================================================
# 4. CLAIM REWRITE — SELF-CONTAINED
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# ------------------------------------------------------------
# 1. Recreate the required target and predictions
# ------------------------------------------------------------

# June actual CTR
y_june = model_df["june_ctr"].astype(float)

# Historical CTR baseline
baseline_june_pred = (
    model_df["hist_ctr"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .values
)

# ------------------------------------------------------------
# 2. Train Random Forest on May and predict June
# ------------------------------------------------------------

features = [
    "hist_impressions",
    "hist_clicks",
    "hist_ctr",
    "hist_avg_position",
    "hist_pageviews",
    "hist_sessions",
    "hist_users",
    "hist_engaged_sessions",
    "hist_scroll_events"
]

X = (
    model_df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y_may = model_df["may_ctr"].astype(float)

from sklearn.ensemble import RandomForestRegressor

rf_honest = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

# Train only using May target
rf_honest.fit(X, y_may)

# Predict June
rf_june_pred = rf_honest.predict(X)

# ------------------------------------------------------------
# 3. Calculate MAE
# ------------------------------------------------------------

baseline_june_mae = mean_absolute_error(
    y_june,
    baseline_june_pred
)

rf_june_mae = mean_absolute_error(
    y_june,
    rf_june_pred
)

# ------------------------------------------------------------
# 4. Calculate relative difference
# ------------------------------------------------------------

relative_difference = (
    (baseline_june_mae - rf_june_mae)
    / baseline_june_mae
) * 100

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

claim_table = pd.DataFrame({
    "metric": [
        "June historical-CTR baseline MAE",
        "June Random Forest MAE",
        "Relative MAE difference (%)"
    ],
    "value": [
        baseline_june_mae,
        rf_june_mae,
        relative_difference
    ]
})

display(claim_table)

# ------------------------------------------------------------
# 6. Safe public-facing claim
# ------------------------------------------------------------

direction = (
    "lower"
    if rf_june_mae < baseline_june_mae
    else "higher"
)

print(
    f"Safe claim: In this June forward test, the Random Forest "
    f"measured MAE was {direction} than the historical-CTR baseline "
    f"by {abs(relative_difference):.2f}%. "
    "This is an observed, directional decision-support result, "
    "not a causal claim."
)

,metric,value
0,June historical-CTR baseline MAE,0.005440
1,June Random Forest MAE,0.004764
2,Relative MAE difference (%),12.433165


Safe claim: In this June forward test, the Random Forest measured MAE was lower than the historical-CTR baseline by 12.43%. This is an observed, directional decision-support result, not a causal claim.


## Failure examples and interpretation

The largest errors are reported using pseudonymous IDs only. The IDs are not model features and are not intended to identify clients or pages.

A key failure mode is low-volume content. When a page has very few impressions, one click can produce an extreme observed CTR. A model trained on historical aggregate behavior may predict a moderate CTR while the realized next-month CTR becomes 0 or 1. These cases can dominate absolute-error rankings.

Therefore, large individual errors should be interpreted together with impression volume. The model may be less reliable for sparse observations even when its overall MAE is small.

In [11]:
error_df = model_df[
    ["client_hash_id", "content_hash_id",
     "june_impressions", "june_clicks", "june_ctr"]
].copy()

error_df["predicted_ctr"] = rf_june_pred
error_df["absolute_error"] = (error_df["june_ctr"] - error_df["predicted_ctr"]).abs()
error_df["signed_error"] = error_df["june_ctr"] - error_df["predicted_ctr"]

largest_errors = error_df.sort_values(
    "absolute_error", ascending=False
).head(10).copy()

largest_errors["volume_bucket"] = pd.cut(
    largest_errors["june_impressions"],
    bins=[-np.inf, 10, 100, 1000, np.inf],
    labels=["<=10", "11-100", "101-1000", "1000+"]
)

display(largest_errors[
    ["june_impressions", "june_clicks", "june_ctr",
     "predicted_ctr", "absolute_error", "volume_bucket"]
])

print("Largest-error cases shown without client/content identifiers.")

,june_impressions,june_clicks,june_ctr,predicted_ctr,absolute_error,volume_bucket
103431,1.0,1.0,1.0,0.000123,0.999877,<=10
75288,1.0,1.0,1.0,0.000528,0.999472,<=10
119172,1.0,1.0,1.0,0.000763,0.999237,<=10
119544,1.0,1.0,1.0,0.001028,0.998972,<=10
4875,1.0,1.0,1.0,0.001072,0.998928,<=10
109641,1.0,1.0,1.0,0.001193,0.998807,<=10
76813,1.0,1.0,1.0,0.001263,0.998737,<=10
134366,1.0,1.0,1.0,0.001425,0.998575,<=10
62372,1.0,1.0,1.0,0.001598,0.998402,<=10
89778,1.0,1.0,1.0,0.001634,0.998366,<=10


Largest-error cases shown without client/content identifiers.


In [12]:
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": rf_honest.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

display(feature_importance)

overprediction_rate = (rf_june_pred > y_june).mean()
underprediction_rate = (rf_june_pred < y_june).mean()

print(f"Overprediction rate: {overprediction_rate:.2%}")
print(f"Underprediction rate: {underprediction_rate:.2%}")
print("Feature importance is descriptive of this fitted Random Forest; it does not establish causality.")

,feature,importance
0,hist_ctr,0.398076
1,hist_avg_position,0.328825
2,hist_impressions,0.156331
3,hist_clicks,0.052028
4,hist_scroll_events,0.021883
5,hist_pageviews,0.021110
6,hist_users,0.009417
7,hist_sessions,0.009112
8,hist_engaged_sessions,0.003218


Overprediction rate: 74.10%
Underprediction rate: 25.90%
Feature importance is descriptive of this fitted Random Forest; it does not establish causality.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.